# EasyRAG Vector Index Basics

## Chapter position

This chapter turns prepared documents into retrieval infrastructure. Chunk boundaries, embedding inputs, vector storage, and workspace artifacts all start here.

## Learning question

How does EasyRAG choose between dense vector search and token fallback, and what does that mean for retrieval behavior?

## Success criteria

- compare fallback token search, dense embedding search, and optional HNSW search
- inspect backend names and search results directly from storage classes
- connect vector-backend behavior to `QueryResult.metadata`

## Source code anchors

- `easyrag.rag.storage.local.TokenVectorStorage`
- `easyrag.rag.storage.local.EmbeddingVectorStorage`
- `easyrag.rag.retrieval.hydration.detect_vector_backend`
- `tests.test_rag_retriever.test_hnsw_backend_is_preferred_when_available`


## Method principles

- `easyrag.rag.storage.local.TokenVectorStorage`: This is the lexical fallback index. It scores records by token overlap when dense embeddings are unavailable or unsuitable.
- `easyrag.rag.storage.local.EmbeddingVectorStorage`: This is the local dense-vector storage implementation. It stores embeddings and serves nearest-neighbor search, optionally with a better backend like HNSW when available.
- `easyrag.rag.retrieval.hydration.detect_vector_backend`: This helper infers which retrieval backend the current result most likely came from. It is an observability label, not a retrieval step by itself.
- `tests.test_rag_retriever.test_hnsw_backend_is_preferred_when_available`: This regression test documents backend preference as executable spec. It checks that the faster HNSW path wins when the optional dependency is installed.


## How the code fits together

The flow in this notebook is stored vectors -> backend detection -> dense search vs token fallback. The goal is not to show every internal detail at once. The goal is to keep the boundary for this stage visible enough that later behavior still feels explainable. If a result looks odd, the intermediate objects in this notebook should tell you where to look next.

In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)

import numpy as np
from easyrag.rag.storage.local import EmbeddingVectorStorage, TokenVectorStorage

## What this cell is proving

We will use the storage classes directly first. That keeps the backend differences visible before we wrap them in `EasyRAG`.

In [ ]:
vector_tmp = tempfile.TemporaryDirectory()
token_store = TokenVectorStorage(vector_tmp.name, "vector-demo")
dense_store = EmbeddingVectorStorage(vector_tmp.name, "vector-demo")
dense_store.set_embedding_func(_stub_embedding)
run_sync(token_store.initialize())
run_sync(dense_store.initialize())

items = [
    {
        "id": "chunk::retrieval",
        "text": "query rewrite improves grounded retrieval",
        "metadata": {"doc_id": "doc::retrieval"},
    },
    {
        "id": "chunk::storage",
        "text": "packed evidence stays available for answer synthesis",
        "metadata": {"doc_id": "doc::storage"},
    },
]
run_sync(token_store.upsert("chunk", items))
run_sync(dense_store.upsert("chunk", items))

token_hits = run_sync(token_store.similarity_search("chunk", "query rewrite", 2))
dense_hits = run_sync(dense_store.similarity_search("chunk", "query rewrite", 2))
_print_json(
    {
        "token_backend": token_store.get_backend_name(),
        "token_hits": token_hits,
        "dense_backend": dense_store.get_backend_name(),
        "dense_hits": dense_hits,
    }
)

## Why this output looks like this

The next cell adds a fake HNSW implementation so the notebook can deterministically show the backend preference order tested in the repo: HNSW first, dense second, token fallback third.

In [ ]:
class FakeHNSWIndex:
    def __init__(self, space: str, dim: int) -> None:
        self.space = space
        self.dim = dim
        self._embeddings = np.zeros((0, dim), dtype=np.float32)
        self._labels = np.zeros((0,), dtype=np.int32)

    def init_index(self, max_elements: int, ef_construction: int, M: int) -> None:
        del max_elements, ef_construction, M

    def add_items(self, embeddings: np.ndarray, labels: np.ndarray) -> None:
        self._embeddings = np.array(embeddings, dtype=np.float32)
        self._labels = np.array(labels, dtype=np.int32)

    def set_ef(self, ef: int) -> None:
        del ef

    def knn_query(self, query_embedding: np.ndarray, k: int):
        normalized_records = self._embeddings / np.maximum(
            np.linalg.norm(self._embeddings, axis=1, keepdims=True), 1e-12
        )
        normalized_query = query_embedding / np.maximum(
            np.linalg.norm(query_embedding, axis=1, keepdims=True), 1e-12
        )
        scores = normalized_records @ normalized_query[0]
        ranked = np.argsort(scores)[::-1][:k]
        distances = 1.0 - scores[ranked]
        return self._labels[ranked][None, :], distances[None, :]


with mock.patch("easyrag.rag.storage.local.hnswlib", mock.Mock(Index=FakeHNSWIndex)):
    hnsw_store = EmbeddingVectorStorage(vector_tmp.name, "hnsw-demo")
    hnsw_store.set_embedding_func(_stub_embedding)
    run_sync(hnsw_store.initialize())
    run_sync(hnsw_store.upsert("chunk", items))
    hnsw_hits = run_sync(hnsw_store.similarity_search("chunk", "query rewrite", 2))
    _print_json({"backend": hnsw_store.get_backend_name(), "hits": hnsw_hits})
    run_sync(hnsw_store.finalize())

## What this cell is proving

The provider-backed path should preserve the same contract even when the backend behavior is less deterministic.

If OpenAI-compatible settings are available, the optional cell builds a real `EasyRAG` workspace and shows which backend label ends up in retrieval metadata.

In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_tmp = tempfile.TemporaryDirectory()
    provider_rag = EasyRAG(
        working_dir=provider_tmp.name, workspace="provider-vector-demo"
    )
    try:
        run_sync(provider_rag.initialize_storages())
        run_sync(
            provider_rag.ainsert(
                "# Retrieval\nQuery rewrite improves grounded retrieval.",
                ids=["doc::retrieval"],
                file_paths=["docs/retrieval.md"],
            )
        )
        provider_result = run_sync(
            provider_rag.aquery(
                "query rewrite",
                QueryParam(mode="naive", rewrite_enabled=False, mqe_enabled=False),
            )
        )
        _print_json(provider_result.metadata)
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")
    finally:
        try:
            run_sync(provider_rag.finalize_storages())
        finally:
            provider_tmp.cleanup()

## Common mistakes / debugging cues

- Do not tune retrieval before you understand chunk boundaries, embedding inputs, and backend choice.
- Watch metadata such as `chunk_strategy`, `vector_backend`, and workspace artifacts, not only final query hits.
- Indexing bugs often look like retrieval bugs until you inspect the payloads that were actually written.

## Takeaway

The retrieval backend determines how similarity is computed, but the interface stays stable. That is the important design point: `EasyRAG.aquery()` can keep returning grounded citations and backend metadata even when the storage implementation changes underneath it.

In [ ]:
run_sync(token_store.finalize())
run_sync(dense_store.finalize())
vector_tmp.cleanup()
print("Cleaned up the vector-index workspace.")

## Where to go next

- Continue with [03_07_build_index_pipeline.ipynb](03_07_build_index_pipeline.ipynb) to see vector indexing inside the full insert pipeline.
- Read [principles/12-vector-index-principles.md](../../docs/principles/12-vector-index-principles.md) for the conceptual view of index backends.
- Revisit [03_05_embedding_inputs_and_provider_behavior.ipynb](03_05_embedding_inputs_and_provider_behavior.ipynb) if you want the provider-input view first.